In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ImagePipeline-Resize") \
    .master("yarn") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print("SparkSession 已创建，运行模式：YARN")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 18:27:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 18:27:19 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


SparkSession 已创建，运行模式：YARN


In [2]:
# Cell 2：读取 HDFS 上的图片
INPUT_PATH = "/user/root/image-pipeline/input/*.jpg"

df = spark.read.format("binaryFile").load(INPUT_PATH)

# 记录处理前的数量
total_before = df.count()
print(f"读取到 {total_before} 张图片")

# 看一眼数据长什么样
df.select("path", "length").show(5, truncate=False)


读取到 500 张图片
+---------------------------------------------------------------+------+
|path                                                           |length|
+---------------------------------------------------------------+------+
|hdfs://master:9000/user/root/image-pipeline/input/cat_10073.jpg|716638|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10173.jpg|518454|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10058.jpg|199433|
|hdfs://master:9000/user/root/image-pipeline/input/dog_1017.jpg |174670|
|hdfs://master:9000/user/root/image-pipeline/input/cat_1002.jpg |170663|
+---------------------------------------------------------------+------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType
import cv2
import numpy as np
from PIL import Image
import io

# 定义清晰度 UDF
def compute_sharpness(binary_data):
    img = Image.open(io.BytesIO(binary_data))
    img_np = np.array(img)
    
    # 判断通道数，再决定要不要转灰度
    if len(img_np.shape) == 3 and img_np.shape[2] == 3:
        # 彩色图（3通道）→ 转灰度
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    else:
        # 已经是灰度图（1通道）→ 直接用
        gray = img_np
    
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


sharpness_udf = udf(compute_sharpness, FloatType())

# 添加清晰度列
df = df.withColumn("sharpness", sharpness_udf("content"))

# 分析分布（看一眼，确认数据正常）
#df.select("sharpness").describe().show()

SHARPNESS_THRESHOLD = 20

df_clean = df.filter(col("sharpness") > SHARPNESS_THRESHOLD)

after_sharpness_filter = df_clean.count()
removed = df.count() - after_sharpness_filter

print(f"清晰度过滤前: {df.count()} 张")
print(f"清晰度过滤后: {after_sharpness_filter} 张")
print(f"过滤掉: {removed} 张（阈值={SHARPNESS_THRESHOLD}）")


清晰度过滤前: 500 张
清晰度过滤后: 495 张
过滤掉: 5 张（阈值=20）


In [6]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, IntegerType
from PIL import Image
import imagehash
import io

# ===== 第1步：计算感知哈希 =====
def compute_phash(binary_data):
    """输入图片字节，输出十六进制感知哈希字符串"""
    img = Image.open(io.BytesIO(binary_data))
    return str(imagehash.phash(img))

phash_udf = udf(compute_phash, StringType())
df_clean = df_clean.withColumn("phash", phash_udf("content"))

print(f"已为 {df_clean.count()} 张图片计算感知哈希")
df_clean.select("path", "phash").show(3, truncate=False)

# ===== 第2步：自连接 + 汉明距离 =====
def hamming_distance(h1, h2):
    """两个十六进制哈希字符串的汉明距离（不同位的个数）"""
    hash1_int = int(h1, 16)
    hash2_int = int(h2, 16)
    return bin(hash1_int ^ hash2_int).count('1')

hamming_udf = udf(hamming_distance, IntegerType())

# 自连接：a.path < b.path 保证不自己配自己、不重复配对
pairs = df_clean.alias("a") \
    .join(df_clean.alias("b"), col("a.path") < col("b.path")) \
    .select(
        col("a.path").alias("image_a"),
        col("a.phash").alias("hash_a"),
        col("b.path").alias("image_b"),
        col("b.phash").alias("hash_b"),
    ) \
    .withColumn("distance", hamming_udf(col("hash_a"), col("hash_b")))

# 筛选重复：汉明距离 ≤ 10
DUPLICATE_THRESHOLD = 10
duplicates = pairs.filter(col("distance") <= DUPLICATE_THRESHOLD)

dup_count = duplicates.count()
print(f"\n发现 {dup_count} 对重复图片")

if dup_count > 0:
    duplicates.select("image_a", "image_b", "distance").show(truncate=False)
else:
    print("✅ 无重复图片，数据集去重通过！")


已为 495 张图片计算感知哈希
+---------------------------------------------------------------+----------------+
|path                                                           |phash           |
+---------------------------------------------------------------+----------------+
|hdfs://master:9000/user/root/image-pipeline/input/cat_10073.jpg|ec63c71c9a90b993|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10173.jpg|c5603e58d33f486b|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10058.jpg|c13bcce01be473d1|
+---------------------------------------------------------------+----------------+
only showing top 3 rows



[Stage 23:=============================================>          (13 + 2) / 16]


发现 0 对重复图片
✅ 无重复图片，数据集去重通过！


In [7]:
from pyspark.sql.types import BinaryType

def resize_to_224(binary_data):
    """
    输入：原始图片字节
    输出：Resize 后 224×224 的 JPEG 字节
    """
    img = Image.open(io.BytesIO(binary_data))
    
    # 如果原始图不是 RGB（比如灰度图、RGBA），统一转成 RGB
    if img.mode != "RGB":
        img = img.convert("RGB")
    
    # Lanczos 插值缩小到 224×224
    img = img.resize((224, 224), Image.LANCZOS)
    
    # 把处理后的图片编码回字节（JPEG 格式）
    output = io.BytesIO()
    img.save(output, format="JPEG", quality=95)
    
    return output.getvalue()

resize_udf = udf(resize_to_224, BinaryType())
df_final = df_clean.withColumn("image_data", resize_udf("content"))

after_resize = df_final.count()
print(f"Resize 完成: {after_resize} 张图片已统一为 224×224")


[Stage 26:===================================>                    (10 + 2) / 16]

Resize 完成: 495 张图片已统一为 224×224


In [10]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 从路径中提取 cat/dog 标签
def extract_label(path):
    filename = path.split("/")[-1]
    return filename.split("_")[0]

extract_label_udf = udf(extract_label, StringType())
df_final = df_final.withColumn("label", extract_label_udf("path"))

# 保存最终结果
OUTPUT_PATH = "/user/root/image-pipeline/output/cleaned_images_224.parquet"

df_final.select("path", "label", "sharpness", "phash", "image_data") \
    .write \
    .mode("overwrite") \
    .parquet(OUTPUT_PATH)

print(f"✅ 最终 Parquet 已保存到: {OUTPUT_PATH}")

# 验证：读回来看看
verify_df = spark.read.parquet(OUTPUT_PATH)
print(f"\n验证读取: {verify_df.count()} 条记录")
verify_df.select("path", "label", "sharpness", "phash").show(5, truncate=False)


✅ 最终 Parquet 已保存到: /user/root/image-pipeline/output/cleaned_images_224.parquet

验证读取: 495 条记录
+---------------------------------------------------------------+-----+---------+----------------+
|path                                                           |label|sharpness|phash           |
+---------------------------------------------------------------+-----+---------+----------------+
|hdfs://master:9000/user/root/image-pipeline/input/dog_10035.jpg|dog  |1055.3376|ea2e2222ab2bebc9|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10135.jpg|dog  |3071.779 |849bd3361f8c1eb1|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10180.jpg|dog  |4165.9946|9472b29c9df74099|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10112.jpg|cat  |135.72612|93c9753925330ee6|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10191.jpg|cat  |875.44275|d6e936cc2156495e|
+---------------------------------------------------------------+-----+---------+----------------+
only showing to

In [13]:
from pyspark.sql.functions import col

total_original = 500
total_after_sharpness = after_sharpness_filter
total_final = verify_df.count()

print("=" * 50)
print("   图像清洗管道 - 处理摘要")
print("=" * 50)
print(f"  原始图片数量:        {total_original}")
print(f"  清晰度过滤后:        {total_after_sharpness}  (过滤 {total_original - total_after_sharpness} 张)")
print(f"  去重后:              {total_after_sharpness}  (去除 0 对重复)")
print(f"  最终保存:            {total_final} 张 (224×224, Parquet)")
print("=" * 50)

print("\n猫狗分布:")
verify_df.groupBy("label").count().orderBy("label").show()

print("\n清晰度分布:")
verify_df.select("sharpness").describe().show()


   图像清洗管道 - 处理摘要
  原始图片数量:        500
  清晰度过滤后:        495  (过滤 5 张)
  去重后:              495  (去除 0 对重复)
  最终保存:            495 张 (224×224, Parquet)

猫狗分布:
+-----+-----+
|label|count|
+-----+-----+
|  cat|  248|
|  dog|  247|
+-----+-----+


清晰度分布:
+-------+------------------+
|summary|         sharpness|
+-------+------------------+
|  count|               495|
|   mean|1207.7241283917667|
| stddev| 4166.035399777683|
|    min|         22.720016|
|    max|          60809.33|
+-------+------------------+



In [15]:
spark.stop()